In [2]:
import numpy as np
import pandas as pd

from md_Helpers import (
    ThermalizationConfig,
    run_thermalization,
)


# ============================================================
# Scan ranges — endpoints are included
# ============================================================

kT_values = np.round(
    np.arange(0.7, 1.0 + 0.01, 0.02),
    decimals=10,
)

rho_values = np.round(
    np.arange(0.5, 0.6 + 0.01, 0.02),
    decimals=10,
)


# ============================================================
# Core thermalization settings
# ============================================================

n_cells = 40
nsteps = 100_000
dt = 0.002

# HDF5 log interval in simulation steps.
nlog = 1_000

seed = 1



# ============================================================
# Execution controls
# ============================================================

base_notes = (
    "Thermalization grid scan over temperature and density"
)

# If True, immediately stop when any grid point fails.
# If False, record the failure and continue through the grid.
stop_on_error = False


# ============================================================
# Validate the requested grid before starting
# ============================================================

if len(kT_values) == 0:
    raise ValueError("kT_values cannot be empty.")

if len(rho_values) == 0:
    raise ValueError("rho_values cannot be empty.")

if np.any(kT_values <= 0):
    raise ValueError("Every kT value must be positive.")

if np.any(rho_values <= 0):
    raise ValueError("Every density must be positive.")

if n_cells <= 0:
    raise ValueError("n_cells must be positive.")

if nsteps <= 0:
    raise ValueError("nsteps must be positive.")

if nlog <= 0:
    raise ValueError("nlog must be positive.")

if dt <= 0:
    raise ValueError("dt must be positive.")

# The workflow requires at least 41 evolved log points for
# its five terminal phase-analysis frames.
number_of_evolved_logs = int(np.ceil(nsteps / nlog))

if number_of_evolved_logs < 41:
    raise ValueError(
        "This workflow requires at least 41 evolved log points. "
        f"The current settings provide only "
        f"{number_of_evolved_logs}. Increase nsteps or reduce nlog."
    )

number_of_runs = len(kT_values) * len(rho_values)
number_of_particles = 4 * n_cells**3

print("Thermalization scan")
print(f"  kT values:       {kT_values.tolist()}")
print(f"  density values:  {rho_values.tolist()}")
print(f"  grid size:       {number_of_runs} runs")
print(f"  particles/run:   {number_of_particles:,}")
print(f"  steps/run:       {nsteps:,}")
print(f"  dt:              {dt:g}")
print(f"  log period:      {nlog:,}")
print()


# ============================================================
# Run every kT-density combination
# ============================================================

results = []
scan_index = 0

for kT in kT_values:
    for rho in rho_values:
        scan_index += 1

        kT = float(kT)
        rho = float(rho)

        print(
            f"[{scan_index:>2}/{number_of_runs}] "
            f"Starting kT={kT:.3f}, rho={rho:.3f}"
        )

        config = ThermalizationConfig(
            n_fcc_cells=n_cells,
            target_rho=rho,
            nsteps=nsteps,
            kT=kT,
            dt=dt,
            log_period=nlog,
            seed=seed,
        )

        try:
            result = run_thermalization(config)

            created_new = bool(
                result.get("created_new", False)
            )
            skipped = bool(result.get("skipped", False))

            if created_new:
                action = "created"
            elif skipped:
                action = "reused"
            else:
                action = "returned"

            print(
                f"    {action}: "
                f"Run_ID={result['run_id']}, "
                f"status={result.get('status', 'unknown')}"
            )

            results.append({
                "scan_index": scan_index,
                "kT": kT,
                "rho": rho,
                "run_id": result["run_id"],
                "action": action,
                "status": result.get("status"),
                "created_new": created_new,
                "skipped_existing": skipped,
                "run_signature": result.get(
                    "run_signature"
                ),
                "error": None,
            })

        except Exception as error:
            print(
                f"    FAILED: "
                f"{type(error).__name__}: {error}"
            )

            results.append({
                "scan_index": scan_index,
                "kT": kT,
                "rho": rho,
                "run_id": None,
                "action": "failed",
                "status": "Failed",
                "created_new": False,
                "skipped_existing": False,
                "run_signature": config.run_signature,
                "error": (
                    f"{type(error).__name__}: {error}"
                ),
            })

            if stop_on_error:
                raise


# ============================================================
# Display the completed scan
# ============================================================

thermalization_scan = (
    pd.DataFrame(results)
    .sort_values(["kT", "rho"])
    .reset_index(drop=True)
)

print()
print("Thermalization scan results:")
display(thermalization_scan)

print()
print("Run-ID grid:")

run_id_grid = thermalization_scan.pivot(
    index="kT",
    columns="rho",
    values="run_id",
)

display(run_id_grid)

failed_runs = thermalization_scan.loc[
    thermalization_scan["action"] == "failed"
]

if failed_runs.empty:
    print(
        f"All {number_of_runs} grid points completed "
        "or reused an existing matching run."
    )
else:
    print(
        f"{len(failed_runs)} of {number_of_runs} "
        "grid points failed:"
    )
    display(
        failed_runs[
            ["kT", "rho", "error"]
        ]
    )

Thermalization scan
  kT values:       [0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
  density values:  [0.5, 0.52, 0.54, 0.56, 0.58, 0.6]
  grid size:       96 runs
  particles/run:   256,000
  steps/run:       100,000
  dt:              0.002
  log period:      1,000

[ 1/96] Starting kT=0.700, rho=0.500
    reused: Run_ID=20260914185855, status=Complete
[ 2/96] Starting kT=0.700, rho=0.520
    reused: Run_ID=20260914190258, status=Complete
[ 3/96] Starting kT=0.700, rho=0.540
    reused: Run_ID=20260914190403, status=Complete
[ 4/96] Starting kT=0.700, rho=0.560
    reused: Run_ID=20260914190612, status=Complete
[ 5/96] Starting kT=0.700, rho=0.580
    reused: Run_ID=20260914190653, status=Complete
[ 6/96] Starting kT=0.700, rho=0.600
    reused: Run_ID=20260914190819, status=Complete
[ 7/96] Starting kT=0.720, rho=0.500
    reused: Run_ID=20260915160453, status=Complete
[ 8/96] Starting kT=0.720, rho=0.520
    reused: Run_ID=2026091516

,scan_index,kT,rho,run_id,action,status,created_new,skipped_existing,run_signature,error
0,1,0.7,0.50,20260914185855,reused,Complete,False,True,5a410daef8ddadef3e127c117f389bec19246e70564f78...,None
1,2,0.7,0.52,20260914190258,reused,Complete,False,True,e94e1a324d09ea05b6dcc1bc2ceb3a547d660a3270559b...,None
2,3,0.7,0.54,20260914190403,reused,Complete,False,True,73e746d015397842b799a6c83181e54eff00e79130ad79...,None
3,4,0.7,0.56,20260914190612,reused,Complete,False,True,61994efc9c492755d3b8b9c4f2efd6c2a852ebfc6fb676...,None
4,5,0.7,0.58,20260914190653,reused,Complete,False,True,b6c821e2453b75499857e3b292a78f4cab6ec89bc45463...,None
...,...,...,...,...,...,...,...,...,...,...
91,92,1.0,0.52,20260915184318,reused,Running,False,True,d923251aa1a68a9e447dc55e31aa789974343c1f8a4737...,None
92,93,1.0,0.54,20260915184356,created,Complete,True,False,ccfee852b895299e91b051961b7602053cc50eefc3d215...,None
93,94,1.0,0.56,20260915184510,reused,Complete,False,True,e14f7761d0f13af24f455c4730171199800afe2ddbec8f...,None
94,95,1.0,0.58,20260915184704,reused,Complete,False,True,0a95937a28f75b2ae6e74ecd636205a2299ce5574e6673...,None



Run-ID grid:


rho,0.50,0.52,0.54,0.56,0.58,0.60
kT,,,,,,
0.70,20260914185855,20260914190258,20260914190403,20260914190612,20260914190653,20260914190819
0.72,20260915160453,20260915160701,20260915160910,20260915161116,20260915161324,20260915161529
0.74,20260915161736,20260915161946,20260915162155,20260915162403,20260915162612,20260915162817
0.76,20260915163026,20260915163230,20260915163439,20260915163643,20260915163848,20260915164052
0.78,20260915164258,20260915164502,20260915164705,20260915164908,20260915165112,20260915165316
0.80,20260915165520,20260915165726,20260915165929,20260915170133,20260915170337,20260915170540
0.82,20260915170743,20260915170947,20260915171151,20260915171354,20260915171557,20260915171800
0.84,20260915172005,20260915172207,20260915172410,20260915172615,20260915172819,20260915173021
0.86,20260915173225,20260915173427,20260915173627,20260915173827,20260915174027,20260915174228


All 96 grid points completed or reused an existing matching run.
